Importa da Assets (backend REST, nessuna autenticazione richiesta) i dati maestri e lo storico ordini gia' esistenti nel sistema gestionale:
- clienti, articoli: dimensioni di riferimento (fonte di verita')
- ordini storici + dettagli ordine: storico da unire, in Gold, con i nuovi ordini estratti dai PDF (silver_ordini / silver_ordini_righe)

Le tabelle storiche vengono scritte con lo stesso "shape" delle tabelle Silver prodotte dal flusso PDF (02_bronze_to_silver), cosi' il notebook
Gold puo' fare semplicemente UNION delle due fonti senza logica differenziata.

"Pulizia": qui i dati arrivano gia' dal sistema gestionale (non da OCR/LLM su PDF), quindi non serve un matching fuzzy - ma applichiamo comunque controlli minimi di integrita' (righe con cliente/articolo non risolto, quantita'/prezzi non plausibili) per coerenza con la pipeline dei nuovi documenti, e per intercettare eventuali incongruenze nei dati storici.

In [1]:
BACKEND_BASE_URL = "http://ja.4labs.it:8080"
 
import time
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
 
 
def _request_con_retry(metodo: str, url: str, max_tentativi: int = 4, attesa_iniziale: float = 3.0, **kwargs) -> requests.Response:
    attesa = attesa_iniziale
    ultimo_errore = None
    for tentativo in range(1, max_tentativi + 1):
        try:
            resp = requests.request(metodo, url, timeout=15, **kwargs)
            if resp.status_code == 503:
                print(f"  [avviso] 503 (tentativo {tentativo}/{max_tentativi}), riprovo in {attesa:.0f}s...")
                ultimo_errore = requests.exceptions.HTTPError(f"503 su {url}")
                time.sleep(attesa)
                attesa *= 2
                continue
            return resp
        except requests.exceptions.ConnectionError as e:
            print(f"  [avviso] errore di connessione (tentativo {tentativo}/{max_tentativi}), riprovo in {attesa:.0f}s...")
            ultimo_errore = e
            time.sleep(attesa)
            attesa *= 2
    raise RuntimeError(f"Impossibile contattare '{url}' dopo {max_tentativi} tentativi. Ultimo errore: {ultimo_errore}")
 
 
def get_json(path: str):
    resp = _request_con_retry("GET", f"{BACKEND_BASE_URL}{path}")
    resp.raise_for_status()
    return resp.json()

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 3, Finished, Available, Finished, False)

In [2]:
# Dimensioni: clienti e articoli (fonte di verita', importati cosi' come sono)
 
print("Scarico clienti da Assets...")
clienti_json = get_json("/api/clienti")
print(f"  {len(clienti_json)} clienti.")
 
print("Scarico articoli da Assets...")
articoli_json = get_json("/api/articoli")
print(f"  {len(articoli_json)} articoli.")
 
schema_cliente_dim = StructType([
    StructField("id", StringType()),
    StructField("ragioneSociale", StringType()),
    StructField("partitaIva", StringType()),
    StructField("indirizzo", StringType()),
    StructField("cap", StringType()),
    StructField("citta", StringType()),
    StructField("provincia", StringType()),
    StructField("email", StringType()),
    StructField("telefono", StringType()),
])
 
schema_articolo_dim = StructType([
    StructField("codice", StringType()),
    StructField("descrizione", StringType()),
    StructField("categoria", StringType()),
    StructField("unitaMisura", StringType()),
    StructField("prezzoListino", DoubleType()),
])
 
df_clienti_assets = spark.createDataFrame(clienti_json, schema=schema_cliente_dim)
df_articoli_assets = spark.createDataFrame(articoli_json, schema=schema_articolo_dim)
 
df_clienti_assets.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_clienti_assets")
df_articoli_assets.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_articoli_assets")
 
print(f"silver_clienti_assets: {df_clienti_assets.count()} righe")
print(f"silver_articoli_assets: {df_articoli_assets.count()} righe")

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 4, Finished, Available, Finished, False)

Scarico clienti da Assets...
  20 clienti.
Scarico articoli da Assets...
  40 articoli.
silver_clienti_assets: 20 righe
silver_articoli_assets: 40 righe


In [3]:
# Storico ordini: testata + dettagli, dalle liste complete
 
print("\nScarico ordini storici da Assets...")
ordini_json = get_json("/api/ordini")
print(f"  {len(ordini_json)} ordini storici.")
 
print("Scarico dettagli ordine storici da Assets...")
dettagli_json = get_json("/api/dettagli-ordine")
print(f"  {len(dettagli_json)} righe di dettaglio storiche.")

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 5, Finished, Available, Finished, False)


Scarico ordini storici da Assets...
  80 ordini storici.
Scarico dettagli ordine storici da Assets...
  400 righe di dettaglio storiche.


In [4]:
# Trasformo gli ordini storici nello stesso "shape" di silver_ordini, cosi'
# il notebook Gold puo' fare UNION diretta con i nuovi ordini da PDF.
# Lo storico non ha (e non puo' avere) i campi specifici dell'estrazione
# AI (confidenza_match, note_anomalia, intervento_umano_necessario): per
# questi mettiamo valori di default coerenti (storico = gia' validato dal
# gestionale, quindi confidenza "alta" e nessuna anomalia da matching).
 
righe_ordini_storici = []
for o in ordini_json:
    cliente = o.get("cliente") or {}
    righe_ordini_storici.append({
        "pdf_nome": None,  # non proviene da un PDF
        "id_ordine_storico": o.get("idOrdine"),
        "data_elaborazione": None,
        "intervento_umano_necessario": False,
        "motivo_intervento_umano": None,
        "id_cliente": cliente.get("id"),
        "cliente_ragione_sociale": cliente.get("ragioneSociale"),
        "cliente_partita_iva": cliente.get("partitaIva"),
        "cliente_citta": cliente.get("citta"),
        "cliente_provincia": cliente.get("provincia"),
        "cliente_fonte": "backend_verificato",
        "riferimento_ordine": o.get("riferimentoCliente"),
        "data_ordine": o.get("dataOrdine"),
        "data_consegna_richiesta": None,
        "condizioni_pagamento": None,
        "note_generali": None,
        "fonte_dato": "storico_assets",
    })
 
schema_ordine_storico = StructType([
    StructField("pdf_nome", StringType()),
    StructField("id_ordine_storico", StringType()),
    StructField("data_elaborazione", StringType()),
    StructField("intervento_umano_necessario", StringType()),
    StructField("motivo_intervento_umano", StringType()),
    StructField("id_cliente", StringType()),
    StructField("cliente_ragione_sociale", StringType()),
    StructField("cliente_partita_iva", StringType()),
    StructField("cliente_citta", StringType()),
    StructField("cliente_provincia", StringType()),
    StructField("cliente_fonte", StringType()),
    StructField("riferimento_ordine", StringType()),
    StructField("data_ordine", StringType()),
    StructField("data_consegna_richiesta", StringType()),
    StructField("condizioni_pagamento", StringType()),
    StructField("note_generali", StringType()),
    StructField("fonte_dato", StringType()),
])
 
df_ordini_storici = spark.createDataFrame(righe_ordini_storici, schema=schema_ordine_storico)
df_ordini_storici = df_ordini_storici.withColumn("data_ordine", F.to_date(F.col("data_ordine")))
 
df_ordini_storici.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ordini_storici")
print(f"\nsilver_ordini_storici: {df_ordini_storici.count()} righe")

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 6, Finished, Available, Finished, False)


silver_ordini_storici: 80 righe


In [5]:
# Dettagli ordine storici, nello stesso "shape" di silver_ordini_righe
 
righe_dettagli_storici = []
for d in dettagli_json:
    ordine = d.get("ordine") or {}
    articolo = d.get("articolo") or {}
    quantita = d.get("quantita")
    prezzo_unitario = d.get("prezzoUnitario")
    righe_dettagli_storici.append({
        "id_ordine_storico": ordine.get("idOrdine"),
        "id_riga_storico": d.get("idRiga"),
        "codice_dichiarato": articolo.get("codice"),
        "descrizione_dichiarata": d.get("descrizioneArticolo") or articolo.get("descrizione"),
        "codice_articolo_match": articolo.get("codice"),
        "descrizione_match": articolo.get("descrizione"),
        "quantita": float(quantita) if quantita is not None else None,
        "unita_misura": articolo.get("unitaMisura"),
        "prezzo_dichiarato": prezzo_unitario,
        "prezzo_listino_catalogo": articolo.get("prezzoListino"),
        "confidenza_match": "alta",  # storico gia' validato dal gestionale
        "note_anomalia": None,
        "importo_riga_calcolato": d.get("importoRiga"),
        "fonte_dato": "storico_assets",
    })
 
schema_dettaglio_storico = StructType([
    StructField("id_ordine_storico", StringType()),
    StructField("id_riga_storico", IntegerType()),
    StructField("codice_dichiarato", StringType()),
    StructField("descrizione_dichiarata", StringType()),
    StructField("codice_articolo_match", StringType()),
    StructField("descrizione_match", StringType()),
    StructField("quantita", DoubleType()),
    StructField("unita_misura", StringType()),
    StructField("prezzo_dichiarato", DoubleType()),
    StructField("prezzo_listino_catalogo", DoubleType()),
    StructField("confidenza_match", StringType()),
    StructField("note_anomalia", StringType()),
    StructField("importo_riga_calcolato", DoubleType()),
    StructField("fonte_dato", StringType()),
])
 
df_dettagli_storici = spark.createDataFrame(righe_dettagli_storici, schema=schema_dettaglio_storico)
 
# Pulizia minima: segnala righe storiche con riferimenti rotti (articolo o
# ordine mancante nello storico) o quantita'/prezzo non plausibili, per
# trasparenza - non le scarta, solo le marca.
df_dettagli_storici = df_dettagli_storici.withColumn(
    "anomalia_integrita",
    F.when(F.col("codice_articolo_match").isNull(), F.lit("Articolo storico non risolto"))
     .when(F.col("id_ordine_storico").isNull(), F.lit("Riferimento ordine storico mancante"))
     .when((F.col("quantita").isNotNull()) & (F.col("quantita") <= 0), F.lit("Quantita' storica non plausibile"))
     .otherwise(F.lit(None).cast("string"))
)
 
n_anomalie = df_dettagli_storici.filter(F.col("anomalia_integrita").isNotNull()).count()
if n_anomalie > 0:
    print(f"\n[ATTENZIONE] {n_anomalie} righe storiche con anomalie di integrita' (segnalate, non scartate):")
    df_dettagli_storici.filter(F.col("anomalia_integrita").isNotNull()).select(
        "id_ordine_storico", "codice_dichiarato", "anomalia_integrita"
    ).show(20, truncate=False)
 
df_dettagli_storici.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ordini_righe_storiche")
print(f"\nsilver_ordini_righe_storiche: {df_dettagli_storici.count()} righe")

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 7, Finished, Available, Finished, False)


silver_ordini_righe_storiche: 400 righe


In [6]:
print("\nCompletato. Tabelle create/aggiornate:")
for nome_tabella in [
    "silver_clienti_assets", "silver_articoli_assets",
    "silver_ordini_storici", "silver_ordini_righe_storiche",
]:
    print(f"  - {nome_tabella}")

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 8, Finished, Available, Finished, False)


Completato. Tabelle create/aggiornate:
  - silver_clienti_assets
  - silver_articoli_assets
  - silver_ordini_storici
  - silver_ordini_righe_storiche


In [7]:
df = spark.sql("SELECT * FROM LakeHouse.dbo.silver_ordini_righe_storiche LIMIT 1000")
display(df)

StatementMeta(, 64a52c39-3a68-4b24-8125-a3779b83b00f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e1af0c2-475e-4cea-979e-b73579a80742)